In [18]:
from akula import API_TOKEN
#print(API_TOKEN)

In [14]:
import telebot
from telebot import types
import sqlite3
from datetime import datetime
import random



bot = telebot.TeleBot(API_TOKEN)

# Проверка, существует ли пользователь в базе данных
def user_exists(user_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''SELECT COUNT(1) FROM users WHERE user_id = ?''', (user_id,))
    exists = cursor.fetchone()[0] > 0
    conn.close()
    return exists

# Сохранение информации о пользователе в базу данных
def save_user_info(user_info):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    if user_exists(user_info['user_id']):
        cursor.execute('''UPDATE users
                          SET first_name = ?, last_name = ?, username = ?, language_code = ?, is_bot = ?, birth_date = ?, last_activity_date = ?
                          WHERE user_id = ?''',
                       (user_info['first_name'], user_info.get('last_name'), user_info.get('username'),
                        user_info.get('language_code'), user_info['is_bot'], user_info.get('birth_date'),
                        user_info['last_activity_date'], user_info['user_id']))
    else:
        cursor.execute('''INSERT INTO users
                          (user_id, first_name, last_name, username, language_code, is_bot, birth_date, registration_date, last_activity_date)
                          VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)''',
                       (user_info['user_id'], user_info['first_name'], user_info.get('last_name'), user_info.get('username'),
                        user_info.get('language_code'), user_info['is_bot'], user_info.get('birth_date'),
                        user_info['registration_date'], user_info['last_activity_date']))
    conn.commit()
    conn.close()

# Обновление даты последней активности пользователя
def update_last_activity(user_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''UPDATE users 
                      SET last_activity_date = ? 
                      WHERE user_id = ?''',
                   (datetime.now().strftime('%Y-%m-%d %H:%M:%S'), user_id))
    conn.commit()
    conn.close()

# Получение случайного фильма из базы данных
def get_random_movie():
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''SELECT id, name, slogan, description, year FROM movies ORDER BY RANDOM() LIMIT 1''')
    movie = cursor.fetchone()
    conn.close()
    return movie


# Получение URL-адреса превью по ID фильма
def get_preview_url(movie_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''SELECT preview_url FROM posters WHERE movie_id = ?''', (movie_id,))
    preview_url = cursor.fetchone()
    conn.close()
    return preview_url[0] if preview_url else None


# Обработчик команды /start
@bot.message_handler(commands=['start'])
def send_welcome(message):
    user = message.from_user
    user_info = {
        'user_id': user.id,
        'first_name': user.first_name,
        'last_name': user.last_name,
        'username': user.username,
        'language_code': user.language_code,
        'is_bot': user.is_bot,
        'birth_date': None,
        'registration_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'last_activity_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    if user_exists(user.id):
        # Если пользователь уже существует, обновляем только дату последней активности
        update_last_activity(user.id)
    else:
        # Если пользователь новый, сохраняем всю информацию
        save_user_info(user_info)
    
    bot.reply_to(message, f"Привет, {user.first_name}! Добро пожаловать в наш бот для оценки фильмов.")
    send_random_movie(message)

movie_id = None

# Отправка случайного фильма для оценки
def send_random_movie(message):
    global movie_id
    movie = get_random_movie()
    if movie:
        movie_id, title, tagline, description, release_year = movie
        preview_url = get_preview_url(movie_id)
        
        # Создание кнопок для оценки фильма
        reply_markup = types.ReplyKeyboardMarkup(row_width=3, resize_keyboard=True)
        btn_dislike = types.KeyboardButton('👎')
        btn_menu = types.KeyboardButton('📺')
        btn_like = types.KeyboardButton('👍')
        reply_markup.add(btn_dislike, btn_menu, btn_like)
        

        
        # Формирование текста сообщения
        movie_text = f"*{title}* \n\n"
        if tagline:
            movie_text += f"*{tagline}*\n\n"
        if description:
            movie_text += f"{description}\n\n"
        movie_text += f"*Год фильма: {release_year}*"

    if preview_url:
        # Отправка фотографии с inline_markup
        bot.send_photo(message.chat.id, preview_url, caption=movie_text, parse_mode='Markdown', reply_markup=reply_markup)
        
    else:
        bot.send_message(message.chat.id, "Не удалось найти фильм для оценки.")



#Обработка '👎', '👍'
@bot.message_handler(func=lambda message: message.text in ['👎', '👍'])
def movie_rating_handler(message):
    user = message.from_user
    update_last_activity(user.id)  # Обновляем дату последней активности

    rating = None
    want_to_watch = None

    if message.text == '👎':
        want_to_watch = -1
        bot.reply_to(message, "Вы поставили отрицательную оценку.")
    elif message.text == '👍':
        want_to_watch = 1
        bot.reply_to(message, "Вы поставили положительную оценку.")

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    cursor.execute('''INSERT INTO actions
                      (user_id, movie_id, want_to_watch, rating)
                      VALUES (?, ?, ?, ?)''',
                   (user.id, movie_id, want_to_watch, rating))

    conn.commit()
    conn.close()

    # Отправляем следующий фильм после оценки
    send_random_movie(message)
#Обработка '📺'
@bot.message_handler(func=lambda message: message.text == '📺')
def show_liked_movies(message):
    user = message.from_user
    update_last_activity(user.id)  # Обновляем дату последней активности

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Получаем список фильмов, на которые пользователь поставил "👍"
    cursor.execute('''SELECT movies.name, movies.year
                      FROM movies
                      JOIN actions ON movies.id = actions.movie_id
                      WHERE actions.user_id = ? AND actions.want_to_watch = 1''',
                   (user.id,))

    liked_movies = cursor.fetchall()
    conn.close()

    if liked_movies:
        # Формируем список фильмов в виде строки
        movies_list = "\n".join([f"{title} ({year})" for title, year in liked_movies])
        bot.reply_to(message, f"Список фильмов, на которые вы поставили 👍 :\n\n{movies_list}")
    else:
        bot.reply_to(message, "Вы еще не поставили 👍 ни одному фильму.")
        send_random_movie(message)

#Обрабочик /drop
@bot.message_handler(commands=['drop'])
def drop_user_data(message):
    user = message.from_user

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Удаляем все записи пользователя из таблицы "actions"
    cursor.execute('''DELETE FROM actions WHERE user_id = ?''', (user.id,))

    conn.commit()
    conn.close()

    bot.reply_to(message, "Ваши сохраненные списки оценок успешно очищены.")
    send_random_movie(message)
    
# Обработчик некорректного ввода
@bot.message_handler(func=lambda message: message.text not in ['👎', '📺', '👍'])
def handle_incorrect_input(message):
    bot.reply_to(message, "Некорректный ввод. Пожалуйста, используйте кнопки для оценки фильма.")

if __name__ == '__main__':
    bot.polling(none_stop=True)
